# Chicago Menu Time Machine: NYPL importer

Rebuilds `frontend/public/data/menus.json` from the New York Public Library's
[What's on the Menu?](https://menus.nypl.org/) open dataset, on Colab's machine instead of your laptop.

It downloads NYPL's bulk CSV archive (about 37 MB), picks the Chicago menus, joins dishes, prices and page
positions, and writes a compact JSON file. **No scan images are downloaded**: the app loads them from NYPL's
servers when a visitor opens a dish. Runs in a minute or two on the free tier; no GPU needed.

Run the cells top to bottom.

## 1. Get the code
Set `REPO` to your fork if you have one.

In [ ]:
REPO = "https://github.com/sfogle/chicago-menu-time-machine.git"
BRANCH = "main"

import os
if not os.path.isdir("chicago-menu-time-machine"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd chicago-menu-time-machine/pipeline

## 2. Preview the selection
`--dry-run` downloads the dataset and prints the menus it would import, without writing anything.
The importer uses only the Python standard library, so there is nothing to install.

In [ ]:
!python import_nypl_chicago.py --dry-run --check-latest

## 3. Run the import
Options worth knowing:

| Flag | Effect |
| --- | --- |
| `--max-menus 40` | how many menus to keep |
| `--per-venue 2` | most menus for any one venue |
| `--max-dishes 120` | cap per menu, keeps the JSON small |
| `--no-dining-cars` | leave out railroad dining-car menus |
| `--validate-images` | HEAD-check every page image (rate limited to 2 requests a second) and switch to NYPL's web-size image where the larger size is refused |
| `--no-geocode` | never call OpenStreetMap Nominatim |

In [ ]:
!python import_nypl_chicago.py --validate-images

## 4. Look at the result

In [ ]:
import json, collections
doc = json.load(open("../frontend/public/data/menus.json"))
rows = doc["restaurants"]
print(len(rows), "menus,", sum(len(r["menu"]) for r in rows), "dishes")
for decade, n in sorted(collections.Counter(r["year"] // 10 * 10 for r in rows).items()):
    print(f"  {decade}s: {n}")
for r in rows:
    print(f"{r['year']}  {r['name']:<46.46} {len(r['menu']):>4} dishes  {r['address']}")

## 5. Take the file home
Download `menus.json` and drop it into `frontend/public/data/` in your local checkout, then commit it.
(Committing from Colab would need a GitHub token; downloading avoids putting one in a notebook.)

In [ ]:
from google.colab import files
files.download("../frontend/public/data/menus.json")